In [ ]:
from github import Github, Auth
import pandas as pd
from datetime import datetime, timedelta
import time
import os
import itertools

# --- 1. Load AIDev blacklist ---
print("Loading AIDev blacklist...")
all_pr_df = pd.read_parquet("hf://datasets/hao-li/AIDev/all_pull_request.parquet")
all_repo_df = pd.read_parquet("hf://datasets/hao-li/AIDev/all_repository.parquet")

blacklisted_repo_names = set(all_repo_df['full_name'].unique())
blacklisted_user_ids = set(all_pr_df['user_id'].unique())

# --- 2. GitHub API with token rotation ---
token_names = ["GITHUB_TOKEN_1", "GITHUB_TOKEN_2", "GITHUB_TOKEN_3"]
tokens = [os.getenv(name) for name in token_names]
token_cycle = itertools.cycle(tokens)

g = Github(auth=Auth.Token(next(token_cycle)))

# --- 3. Config ---
TARGET_COUNT = 100
MIN_STARS = 50
MIN_YEARS = 3
created_before = (datetime.now() - timedelta(days=MIN_YEARS * 365)).strftime('%Y-%m-%d')

# Agent detection keywords
agent_pr_authors = ["devin-ai-integration[bot]"]
agent_pr_branch_prefixes = ["codex/", "copilot/", "cursor/"]
agent_pr_coauthors = ["Claude"]

print(f"Searching repos created before {created_before} with ≥{MIN_STARS} stars")
query = f"stars:>={MIN_STARS} created:<{created_before} is:public"
search = g.search_repositories(query=query, sort="stars", order="desc")

# --- 4. Iterate safely and save as we go ---
count = 0
with open("true_human_baseline.txt", "w", encoding="utf-8") as f:

    for repo in search:
        if count >= TARGET_COUNT:
            break

        # --- 4a. AIDev blacklist ---
        if repo.full_name in blacklisted_repo_names or repo.owner.id in blacklisted_user_ids:
            continue

        owner_login = (repo.owner.login or "").lower()
        if repo.owner.type == "Bot" or "bot" in owner_login:
            continue

        skip_repo = False

        # --- 4b. Branch names ---
        try:
            for branch in repo.get_branches():
                branch_name = branch.name.lower()
                if any(branch_name.startswith(prefix) for prefix in agent_pr_branch_prefixes):
                    skip_repo = True
                    break
        except Exception:
            pass

        if skip_repo:
            continue

        # --- 4c. PR metadata ---
        try:
            for pr in repo.get_pulls(state="all"):
                # Author filter
                if pr.user.login in agent_pr_authors:
                    skip_repo = True
                    break
                # Branch filter (already checked above)
                if any(pr.head.ref.lower().startswith(prefix) for prefix in agent_pr_branch_prefixes):
                    skip_repo = True
                    break
                # Co-author in PR body or title
                pr_text = (pr.body or "") + " " + (pr.title or "")
                if any(f"Co-Authored-By: {author}" in pr_text for author in agent_pr_coauthors):
                    skip_repo = True
                    break
        except Exception:
            pass

        if skip_repo:
            continue

        # --- 4d. Passed all filters ---
        count += 1
        f.write(
            f"{repo.full_name} | {repo.html_url} | "
            f"0 agent PRs | first agent PR: 2025-06-02 | "
            f"repo creation: {repo.created_at.date()}\n"
        )
        f.flush()
        print(f"[{count}/{TARGET_COUNT}] {repo.full_name}")

        # Rotate token every repo to reduce hitting rate limits
        g = Github(auth=Auth.Token(next(token_cycle)))
        time.sleep(1)  # optional, reduce if needed

print("Done.")

Loading AIDev blacklist...
Searching repos created before 2023-02-27 with ≥50 stars
[1/100] codecrafters-io/build-your-own-x
[2/100] sindresorhus/awesome
[3/100] kamranahmedse/developer-roadmap
[4/100] jwasham/coding-interview-university
[5/100] vinta/awesome-python
[6/100] 996icu/996.ICU
[7/100] awesome-selfhosted/awesome-selfhosted
[8/100] practical-tutorials/project-based-learning
[9/100] torvalds/linux
[10/100] vuejs/vue
[11/100] trimstray/the-book-of-secret-knowledge
[12/100] ossu/computer-science
[13/100] trekhleb/javascript-algorithms
[14/100] getify/You-Dont-Know-JS
[15/100] CyC2018/CS-Notes
[16/100] jackfrued/Python-100-Days
[17/100] github/gitignore
[18/100] massgravel/Microsoft-Activation-Scripts
[19/100] avelino/awesome-go
[20/100] jlevy/the-art-of-command-line
[21/100] Snailclimb/JavaGuide
[22/100] airbnb/javascript
[23/100] 521xueweihan/HelloGitHub
[24/100] ytdl-org/youtube-dl
[25/100] yangshun/tech-interview-handbook
[26/100] labuladong/fucking-algorithm
[27/100] Chalara

In [ ]:
import pandas as pd
import re

# Path to your text file
file_path = r"filtered_repos_3year50pr.txt"

# Read the file
with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

# Extract all "first agent PR: YYYY-MM-DD"
dates_str = re.findall(r"first agent PR:\s*(\d{4}-\d{2}-\d{2})", text)

# Convert to datetime and make a Series
dates = pd.Series(pd.to_datetime(dates_str))

# Compute median
median_date = dates.median()
print("Median first agent PR date:", median_date.date())

Median first agent PR date: 2025-06-02


In [1]:
from github import Github, Auth
import pandas as pd
from datetime import datetime, timedelta
import time
import os
import itertools

# --- 1. Load AIDev blacklist ---
print("Loading AIDev blacklist...")
all_pr_df = pd.read_parquet("hf://datasets/hao-li/AIDev/all_pull_request.parquet")
all_repo_df = pd.read_parquet("hf://datasets/hao-li/AIDev/all_repository.parquet")

blacklisted_repo_names = set(all_repo_df['full_name'].unique())
blacklisted_user_ids = set(all_pr_df['user_id'].unique())

# --- 2. GitHub API with token rotation ---
token_names = ["GITHUB_TOKEN_1", "GITHUB_TOKEN_2", "GITHUB_TOKEN_3"]
tokens = [os.getenv(name) for name in token_names]
token_cycle = itertools.cycle(tokens)

g = Github(auth=Auth.Token(next(token_cycle)))

# --- 3. Config ---
TARGET_COUNT = 100
MIN_STARS = 50
MIN_YEARS = 3

created_before = (datetime.now() - timedelta(days=MIN_YEARS * 365)).strftime('%Y-%m-%d')

# Define BEFORE / AFTER windows
AFTER_START_DATE = datetime(2025, 6, 2)
BEFORE_END_DATE  = AFTER_START_DATE - timedelta(days=1)

# Minimum commits in each period
MIN_COMMITS_BEFORE = 50
MIN_COMMITS_AFTER  = 25

# Agent detection keywords
agent_pr_authors = ["devin-ai-integration[bot]"]
agent_pr_branch_prefixes = ["codex/", "copilot/", "cursor/"]
agent_pr_coauthors = ["Claude"]

print(f"Searching repos created before {created_before} with ≥{MIN_STARS} stars")
query = f"stars:>={MIN_STARS} created:<{created_before} is:public"
search = g.search_repositories(query=query, sort="stars", order="desc")

# --- 4. Iterate safely and save as we go ---
count = 0
with open("true_human_baseline_con.txt", "w", encoding="utf-8") as f:

    for repo in search:
        if count >= TARGET_COUNT:
            break

        # --- 4a. AIDev blacklist ---
        if repo.full_name in blacklisted_repo_names or repo.owner.id in blacklisted_user_ids:
            continue

        owner_login = (repo.owner.login or "").lower()
        if repo.owner.type == "Bot" or "bot" in owner_login:
            continue

        skip_repo = False

        # --- 4b. Branch names ---
        try:
            for branch in repo.get_branches():
                branch_name = branch.name.lower()
                if any(branch_name.startswith(prefix) for prefix in agent_pr_branch_prefixes):
                    skip_repo = True
                    break
        except Exception:
            pass

        if skip_repo:
            continue

        # --- 4c. PR metadata ---
        try:
            for pr in repo.get_pulls(state="all"):
                # Author filter
                if pr.user.login in agent_pr_authors:
                    skip_repo = True
                    break
                # Branch filter (already checked above)
                if any(pr.head.ref.lower().startswith(prefix) for prefix in agent_pr_branch_prefixes):
                    skip_repo = True
                    break
                # Co-author in PR body or title
                pr_text = (pr.body or "") + " " + (pr.title or "")
                if any(f"Co-Authored-By: {author}" in pr_text for author in agent_pr_coauthors):
                    skip_repo = True
                    break
        except Exception:
            pass

        if skip_repo:
            continue

        # --- 4d. Ensure activity in BEFORE and AFTER periods ---
        try:
            # BEFORE activity
            commits_before = repo.get_commits(until=BEFORE_END_DATE)
            before_count = 0
            for _ in commits_before:
                before_count += 1
                if before_count >= MIN_COMMITS_BEFORE:
                    break
            if before_count < MIN_COMMITS_BEFORE:
                continue

            # AFTER activity
            commits_after = repo.get_commits(since=AFTER_START_DATE)
            after_count = 0
            for _ in commits_after:
                after_count += 1
                if after_count >= MIN_COMMITS_AFTER:
                    break
            if after_count < MIN_COMMITS_AFTER:
                continue

        except Exception:
            continue

        # --- 4e. Passed all filters ---
        count += 1
        f.write(
            f"{repo.full_name} | {repo.html_url} | "
            f"0 agent PRs | first agent PR: N/A | "
            f"repo creation: {repo.created_at.date()} | "
            f"before_commits >= {before_count} | "
            f"after_commits >= {after_count}\n"
        )
        f.flush()

        print(f"[{count}/{TARGET_COUNT}] {repo.full_name}")

        # Rotate token every repo to reduce hitting rate limits
        g = Github(auth=Auth.Token(next(token_cycle)))
        time.sleep(1)  # optional, reduce if needed

print("Done.")

Loading AIDev blacklist...
Searching repos created before 2023-02-28 with ≥50 stars


Request GET /repos/codecrafters-io/build-your-own-x/branches failed with 403: Forbidden
Setting next backoff to 1640.020271s
Request GET /repos/codecrafters-io/build-your-own-x/branches failed with 403: Forbidden
Setting next backoff to 0s
Request GET /repos/codecrafters-io/build-your-own-x/branches failed with 403: Forbidden
Setting next backoff to 0s
Request GET /repos/codecrafters-io/build-your-own-x/branches failed with 403: Forbidden
Setting next backoff to 0s
Request GET /repos/codecrafters-io/build-your-own-x/branches failed with 403: Forbidden
Setting next backoff to 0s
Request GET /repos/codecrafters-io/build-your-own-x/branches failed with 403: Forbidden
Setting next backoff to 0s
Request GET /repos/codecrafters-io/build-your-own-x/branches failed with 403: Forbidden
Setting next backoff to 0s
Request GET /repos/codecrafters-io/build-your-own-x/branches failed with 403: Forbidden
Setting next backoff to 0s
Request GET /repos/codecrafters-io/build-your-own-x/branches failed wi

[1/100] codecrafters-io/build-your-own-x
[2/100] kamranahmedse/developer-roadmap
[3/100] vinta/awesome-python
[4/100] awesome-selfhosted/awesome-selfhosted
[5/100] torvalds/linux
[6/100] github/gitignore
[7/100] massgravel/Microsoft-Activation-Scripts
[8/100] avelino/awesome-go
[9/100] Snailclimb/JavaGuide
[10/100] ytdl-org/youtube-dl
[11/100] Chalarangelo/30-seconds-of-code
[12/100] facebook/react-native
[13/100] krahets/hello-algo
[14/100] ripienaar/free-for-dev
[15/100] iptv-org/iptv
[16/100] godotengine/godot
[17/100] tauri-apps/tauri
[18/100] jaywcjlove/awesome-mac
[19/100] Anduin2017/HowToCook
[20/100] mtdvio/every-programmer-should-know
[21/100] puppeteer/puppeteer
[22/100] django/django
[23/100] 3b1b/manim
[24/100] ruanyf/weekly
[25/100] MunGell/awesome-for-beginners
[26/100] realworld-apps/realworld
[27/100] syncthing/syncthing
[28/100] spring-projects/spring-boot
[29/100] anuraghazra/github-readme-stats
[30/100] hoppscotch/hoppscotch
[31/100] tensorflow/models
[32/100] coder/